# CAMB's Boltzmann Physics: From Gauge-Invariant Perturbations to $C_\ell$

The CMB angular power spectrum $C_\ell$ is the 2-point correlator of temperature anisotropies on the sky:

$$
C_\ell \equiv \frac{1}{2\ell+1} \sum_{m=-\ell}^\ell \langle a_{\ell m}^* a_{\ell m} \rangle, \qquad \frac{\Delta T}{T}(\hat{n}) = \sum_{\ell m} a_{\ell m} Y_{\ell m}(\hat{n}). \tag{0}
$$

Audience:
- Cosmology graduate students and researchers who want to understand what CAMB computes internally.

Prerequisites:
- Cosmological perturbation theory at the level of Dodelson (2003) or Baumann (2022).
- Familiarity with $C_\ell$ and spherical harmonics.

Learning goals:
- Understand the full Boltzmann hierarchy for photons, polarization, and neutrinos.
- See how CAMB's line-of-sight integration avoids solving the hierarchy $\ell$-by-$\ell$.
- Extract and visualize CAMB's internal outputs: the visibility function, source functions, and transfer functions.
- Decompose $C_\ell^{TT}$ into Sachs-Wolfe, ISW, and acoustic contributions.
- Vary cosmological parameters and observe their effect on $C_\ell$.

## Outline

CAMB evolves the **Boltzmann equation** for each particle species:

$$
\frac{df}{d\eta} = C[f], \qquad f = f_0(\epsilon) + \delta f(\mathbf{k}, \hat{n}, \eta). \tag{0}
$$

1. **Perturbation variables & gauge choice** — synchronous gauge, metric dofs
2. **Photon Boltzmann hierarchy** — $\Theta_\ell$ equations, Thomson scattering
3. **Polarization hierarchy** — $E$-mode and $B$-mode
4. **Tight coupling & recombination** — diffusion, visibility function
5. **Line-of-sight integration** — CAMB's key innovation
6. **Codebase in action** — live CAMB with extracted internals
7. **Visualizations** — visibility, $C_\ell$ decomposition, $TE$, parameter variation
8. **Interpretation** — what CAMB's physics tells us

---
## Section 1: Perturbation Variables & Gauge Choice

CAMB works in the **synchronous gauge**, where the perturbed FLRW metric is

$$
ds^2 = a(\eta)^2 \left[ -d\eta^2 + (\delta_{ij} + h_{ij})\, dx^i dx^j \right], \tag{1}
$$

with $h_{ij}$ decomposed into scalar, vector, and tensor modes. For scalar perturbations,

$$
h_{ij}(\mathbf{x}, \eta) = \int d^3k\, e^{i\mathbf{k}\cdot\mathbf{x}} \left[ \hat{k}_i\hat{k}_j h(\mathbf{k}, \eta) + \left( \hat{k}_i\hat{k}_j - \frac{1}{3}\delta_{ij} \right) 6\eta(\mathbf{k}, \eta) \right]. \tag{2}
$$

Here $h$ and $\eta$ (not to be confused with conformal time) are the two scalar metric perturbations in synchronous gauge. They satisfy

$$
\dot{h}_{\mathbf{k}} = -3\dot{\eta}_{\mathbf{k}} - 2k\theta_{\mathbf{k}}, \qquad
\theta = 3ik_j \delta T^0_j / (\bar{\rho} + \bar{P}). \tag{3}
$$

The synchronous gauge has the advantage that all perturbation variables are purely "matter-side" — the metric responds to stress-energy via the Einstein equations. CAMB adopts this gauge for all internal computations, converting to **gauge-invariant** variables (the curvature perturbation $\zeta$ or $\mathcal{R}$) only for output.

---
## Section 2: The Photon Boltzmann Hierarchy

### 2.1 Temperature Multipoles

The photon distribution function is expanded in Legendre multipoles $\Theta_\ell$:

$$
\Theta(\mathbf{k}, \hat{n}, \eta) = \sum_{\ell=0}^\infty (-i)^\ell (2\ell+1)\, \Theta_\ell(\mathbf{k}, \eta)\, P_\ell(\hat{k}\cdot\hat{n}). \tag{4}
$$

The Boltzmann equation in synchronous gauge gives the hierarchy (Ma & Bertschinger 1995):

$$
\boxed{
\dot{\Theta}_\ell = \frac{k}{2\ell+1}\left[ \ell\Theta_{\ell-1} - (\ell+1)\Theta_{\ell+1} \right]
- \dot{\tau}\, \Theta_\ell + S_\ell
} \tag{5}
$$

where $\dot{\tau} = -n_e \sigma_T a$ is the differential optical depth. The source terms for the first few multipoles are:

- $\ell = 0$ (monopole): $\dot{\Theta}_0 = -k\Theta_1 - \dot{h}/6$
- $\ell = 1$ (dipole): $\dot{\Theta}_1 = \frac{k}{3}\Theta_0 - \frac{2k}{3}\Theta_2 + \frac{k}{3}\eta - \dot{\tau}\, \Theta_1$
- $\ell = 2$ (quadrupole): $\dot{\Theta}_2 = \frac{2k}{5}\Theta_1 - \frac{3k}{5}\Theta_3 - \dot{\tau}\left[ \Theta_2 - \frac{1}{10}\Pi \right]$

where $\Pi = \Theta_2 + E_2 + E_0$ is the polarization source.

### 2.2 Thomson Scattering Terms

The collision terms are anisotropic because only the quadrupole ($\ell=2$) generates polarization:

$$
C[\Theta_\ell] = -\dot{\tau} \left[ \Theta_\ell - \delta_{\ell 0}\Theta_0 - \delta_{\ell 1} 3i\hat{k}\cdot\vec{v}_b - \frac{1}{10}\Pi\, \delta_{\ell 2} \right]. \tag{6}
$$

Here $\vec{v}_b$ is the baryon velocity. The $\ell \geq 3$ moments are **damped** by scattering but **not sourced** — they decay as $\Theta_{\ell \geq 3} \propto \dot{\tau}^{-1}$.

---
## Section 3: Polarization Hierarchy

Polarization is described by the **Stokes parameters** $Q \pm iU$, expanded in spin-weighted spherical harmonics. The $E$-mode multipoles satisfy (Zaldarriaga & Seljak 1997):

$$
\dot{E}_\ell = \frac{k}{2\ell+1}\left[ \ell E_{\ell-1} - (\ell+1)E_{\ell+1} \right] - \dot{\tau}\,E_\ell + \frac{3}{8}\dot{\tau}\, \Pi\, \delta_{\ell 2}. \tag{7}
$$

The $B$-mode hierarchy is analogous but sourced by tensor modes (no scalar $B$-modes at linear order).

Key physics: polarization is **only generated by quadrupole anisotropy** at last scattering:

$$
E_\ell(k, \eta) = \frac{3}{8} \int_{\eta_{\rm in}}^{\eta_0} d\eta'\, \dot{\tau}(\eta')\, \Pi(k, \eta')\, e^{-\tau(\eta')}\, j_\ell(k(\eta_0-\eta')). \tag{8}
$$

Since the quadrupole $\Theta_2$ is sourced by the **gradient of the dipole** ($\propto k\Theta_1$), and $\Theta_1$ oscillates as $\sin(k r_s)$, polarization peaks are **$\pi/2$ out of phase** with temperature peaks — a direct signature of adiabatic initial conditions.

---
## Section 4: Tight Coupling & Recombination

### 4.1 Tight-Coupling Regime ($z \gtrsim 1100$)

Before recombination, the Thomson scattering rate $\dot{\tau}$ is enormous ($\dot{\tau} \gg k$). The hierarchy simplifies to a **two-fluid system**:

$$
\Theta_1 \approx -\frac{\dot{h}}{3k} - \frac{4\rho_\gamma}{3\rho_b} \frac{\Theta_1 - v_b}{\dots}, \qquad
\Theta_\ell \approx \frac{\ell}{2\ell+1} \frac{k}{\dot{\tau}} \Theta_{\ell-1}\quad (\ell \geq 2). \tag{9}
$$

CAMB solves the **exact hierarchy** but uses the tight-coupling approximation at early times to avoid stiffness, switching to the full equations near recombination.

### 4.2 The Visibility Function

The **visibility function** $g(\eta)$ gives the probability distribution of last scattering:

$$
g(\eta) = -\dot{\tau}(\eta)\, e^{-\tau(\eta)}, \qquad \int_{\eta_{\rm in}}^{\eta_0} g(\eta)\, d\eta = 1. \tag{10}
$$

Reionization at $z \sim 10$ creates a second, broader peak in $g(\eta)$. The $E$-mode spectrum is especially sensitive to the optical depth to reionization $\tau_{\rm reio}$ through the large-angle reionization bump at $\ell \lesssim 20$.

The **comoving sound horizon at recombination** sets the acoustic scale:

$$
r_s(\eta_{\rm rec}) = \int_0^{\eta_{\rm rec}} c_s(\eta')\, d\eta', \qquad
c_s = \frac{1}{\sqrt{3(1+R)}}, \quad R = \frac{3\rho_b}{4\rho_\gamma}. \tag{11}
$$

---
## Section 5: Line-of-Sight Integration (CAMB's Key Innovation)

Naively solving the Boltzmann hierarchy up to $\ell \sim 2500$ would require tracking $\sim 2500$ coupled ODEs per $k$-mode. CAMB avoids this through the **line-of-sight (LOS) integral** (Seljak & Zaldarriaga 1996):

$$
C_\ell^{TT} = \frac{2}{\pi} \int_0^\infty \frac{dk}{k}\, P_{\mathcal{R}}(k)\, |\Delta_\ell^T(k)|^2, \tag{12}
$$

where the **transfer function** projects local physics onto the 2D sky:

$$
\Delta_\ell^T(k) = \int_0^{\eta_0} d\eta\, S_T(k, \eta)\, j_\ell(k(\eta_0-\eta)). \tag{13}
$$

The **source function** $S_T(k, \eta)$ packages all local physics at conformal time $\eta$:

$$
\boxed{
S_T(k, \eta) = g(\eta)\left[ \Theta_0 + \eta + \frac{\dot{\eta}}{k^2} \right]
+ e^{-\tau}\left[ \dot{\eta} + \ddot{h}/6k^2 \right]
+ g(\eta) v_b/k
+ \dots
} \tag{14}
$$

| Term | Physical origin | $\ell$-range |
|------|----------------|-------------|
| $g(\Theta_0 + \eta)$ | Sachs-Wolfe (SW) | $\ell \lesssim 30$ |
| $g(\dot{\eta}/k^2)$ | Doppler shift | $\ell \sim 30$-$200$ |
| $g(\Theta_0 + \eta)(\text{oscillatory})$ | Acoustic oscillations | $\ell \gtrsim 100$ |
| $e^{-\tau}\dot{\eta}$ | Late ISW (dark energy) | $\ell \lesssim 10$ |
| $e^{-\tau}\ddot{h}/6k^2$ | Early ISW (radiation) | $\ell \sim 100$ |

CAMB solves the hierarchy only up to $\ell_{\rm max} \sim 10$-$20$ (enough to capture the source function accurately), then projects using the LOS integral. This reduces the ODE system from $\mathcal{O}(2500)$ to $\mathcal{O}(10)$ variables per $k$-mode — a **$\sim 100\times$ speedup**.

---
## Section 6: Codebase in Action — Live CAMB

We now run CAMB through the codebase wrapper and extract its internal outputs. The core function `camb.get_results()` solves the full Einstein-Boltzmann system:

$$
\text{params} \xrightarrow{\text{camb.get_results}} \{ C_\ell,\; g(\eta),\; \tau(\eta),\; x_e(\eta),\; T_\ell(k) \}. \tag{15}
$$

In [ ]:
import os, sys, json
import numpy as np
from scipy.interpolate import interp1d

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import camb
from scripts.camb_wrapper import (
    compute_cl_camb_powerlaw, _make_camb_params,
)
from scripts.constants import As, k_pivot_phys, T_cmb
from scripts.planck_data import get_planck_data_asymmetric, C_ell_to_d_ell

print(f"CAMB version: {camb.__version__}")
print("All imports OK.")

### 6.1 LCDM Baseline

Compute the angular power spectra $C_\ell^{TT}, C_\ell^{TE}, C_\ell^{EE}$ for Planck 2018 LCDM best-fit parameters $H_0=67.66$, $\Omega_b h^2=0.02242$, $\Omega_c h^2=0.11933$, $\tau=0.054$. The primordial spectrum is a power law:

$$
P_{\mathcal{R}}(k) = A_s \left(\frac{k}{k_{\rm pivot}}\right)^{n_s-1}, \quad A_s=2.1\times10^{-9},\; n_s=0.965,\; k_{\rm pivot}=0.05\,{\rm Mpc}^{-1}. \tag{16}
$$

In [ ]:
ell_max = 2500
params = _make_camb_params(ell_max=ell_max)
params.InitPower.set_params(As=As, ns=0.965, r=0, pivot_scalar=k_pivot_phys)

results = camb.get_results(params)

# C_ell
cls = results.get_cmb_power_spectra(lmax=ell_max)
total = cls['total']
ells = np.arange(2, ell_max + 1)
C_TT = total[ells, 0] / (ells * (ells + 1) / (2 * np.pi))
C_TE = total[ells, 3] / (ells * (ells + 1) / (2 * np.pi))
C_EE = total[ells, 1] / (ells * (ells + 1) / (2 * np.pi))
D_TT = C_TT * ells * (ells + 1) / (2 * np.pi) * (T_cmb * 1e6) ** 2

# Background output (recombination, reionization)
bg = results.get_background_output()
bg_names = bg.dtype.names
print(f"Background columns: {bg_names}")
print(f"C_ell computed: ell=2..{ell_max}")

### 6.2 Extract the Visibility Function

The visibility function $g(\eta) = -\dot{\tau}\, e^{-\tau}$ is the probability density that a CMB photon last scattered at conformal time $\eta$:

$$
g(\eta) = -\dot{\tau}(\eta)\, e^{-\tau(\eta)}, \qquad \langle z_{\rm ls} \rangle = \int_0^\infty g(z)\, z\, dz. \tag{17}
$$

Its width $\Delta z \sim 80$ sets the Silk damping length $k_D^{-1} \propto \sqrt{\Delta \eta_{\rm rec}}$.

In [ ]:
def get_bg_col(bg, candidates):
    for c in candidates:
        if c in bg.dtype.names:
            return bg[c]
    raise KeyError(f"None of {candidates} found in bg columns")

eta_bg = get_bg_col(bg, ['time', 'eta', 'conformal_time'])
vis = get_bg_col(bg, ['vis', 'visibility'])
tau_bg = get_bg_col(bg, ['tau', 'optical_depth', 'opacity'])
x_e = get_bg_col(bg, ['x_e', 'x_electron', 'xe'])

a_bg = get_bg_col(bg, ['scale_factor', 'a'])
z_bg = 1.0 / a_bg - 1.0

rec_idx = int(np.argmax(vis))
print(f"Recombination peak:")
print(f"  z_rec = {z_bg[rec_idx]:.1f}")
print(f"  eta_rec = {eta_bg[rec_idx]:.2f} Mpc")
print(f"  tau(z_rec) = {tau_bg[rec_idx]:.4f}")
print(f"  x_e(z_rec) = {x_e[rec_idx]:.4f}")

reio_mask = (z_bg > 0) & (z_bg < 50) & (vis > 1e-6)
if np.any(reio_mask):
    reio_idx = int(np.argmax(vis[reio_mask]))
    print(f"\nReionization bump:")
    print(f"  z_reio ~ {z_bg[reio_mask][reio_idx]:.1f}")
    print(f"  tau(z=0) = {tau_bg[0]:.4f}")

### 6.3 TE Cross-Correlation Coefficient

The TE correlation coefficient measures the phase relationship between temperature and $E$-mode polarization:

$$
r_\ell^{TE} = \frac{C_\ell^{TE}}{\sqrt{C_\ell^{TT} C_\ell^{EE}}}, \qquad -1 \leq r_\ell^{TE} \leq 1. \tag{18}
$$

For adiabatic modes, $r_\ell^{TE}$ oscillates between $\pm 1$ as crests and nodes alternate.

In [ ]:
rTE = C_TE / np.sqrt(np.clip(C_TT * C_EE, 1e-30, None))
print(f"TE correlation at sample multipoles:")
for ell_val in [2, 10, 30, 100, 200, 500]:
    idx = ell_val - 2
    print(f"  ell={ell_val:3d}:  r_TE = {rTE[idx]:+.4f}")
print(f"\nAnti-correlation trough at ell ~ 330 (acoustic crest vs velocity node)")

---
## Section 7: Visualizations

The line-of-sight integral (Eq.~13) packages all physics into a single projection. We now visualize what CAMB computes internally. The conversion between $C_\ell$ and $D_\ell$ is:

$$
D_\ell^{TT} = \frac{\ell(\ell+1)}{2\pi}\, C_\ell^{TT}\, T_{\rm CMB}^2 \quad [\mu{\rm K}^2]. \tag{19}
$$

Four publication-quality plots using the Tol colorblind palette, 300 DPI, 14pt+ axis labels.

In [ ]:
TOL = {
    "blue": "#4477AA", "red": "#CC3311", "green": "#228833",
    "yellow": "#EE8866", "teal": "#44BB99", "purple": "#AA3377",
    "grey": "#666666", "dark": "#222222",
}
from scripts.plotting import get_path, make_filename
OUT = get_path("diagnostics", "")
os.makedirs(OUT, exist_ok=True)

def _save(fig, name):
    p = os.path.join(OUT, name)
    fig.savefig(p, dpi=300, bbox_inches="tight")
    print(f"  Saved: {p}")
    plt.close(fig)

### Plot 1: Visibility Function $g(z)$

The probability that a CMB photon last scattered at redshift $z$ (Eq.~10):

$$
g(z) = -\frac{d\tau}{dz}\, e^{-\tau(z)}, \qquad \langle z \rangle = \int g(z)\, z\, dz \approx 1090. \tag{P1}
$$

Two peaks: recombination at $z \sim 1090$ and reionization at $z \sim 10$. The finite width $\Delta z \sim 80$ produces **Silk damping** — exponential suppression of power at $\ell \gtrsim 1000$.

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.8))

mask = z_bg < 1500
ax.plot(z_bg[mask], vis[mask], "-", color=TOL["blue"], lw=1.5)
ax.axvline(z_bg[rec_idx], color=TOL["grey"], ls="--", lw=1, alpha=0.5)
ax.text(z_bg[rec_idx] + 30, np.max(vis) * 0.9,
        f"$z_{{\\rm rec}} = {z_bg[rec_idx]:.0f}$",
        fontsize=10, color=TOL["grey"])

ax.set_xlabel(r"$z$", fontsize=14)
ax.set_ylabel(r"$g(z)$ [Mpc$^{-1}$]", fontsize=14)
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.25, which="both")
ax.set_xlim(0, 1450)

fig.tight_layout()
_save(fig, "camb_boltzmann_visibility.png")

### Plot 2: Full-Sky $C_\ell^{TT}$ Decomposition

The full CAMB result for LCDM, computed with two independent methods: `InitPower.set_params` (full Boltzmann hierarchy) and `set_initial_power_table` via `compute_cl_camb_powerlaw`. The difference is negligible at low $\ell$, confirming internal consistency:

$$
\Delta C_\ell = C_\ell^{\rm full} - C_\ell^{\rm pl}, \qquad
C_\ell^{\rm pl} = \text{CAMB power-law with same } A_s, n_s. \tag{20}
$$

In [ ]:
from scripts.camb_wrapper import compute_cl_camb_powerlaw

ells_sw, C_sw, _, _ = compute_cl_camb_powerlaw(ell_max=30)
D_sw = C_ell_to_d_ell(ells_sw, C_sw)
D_full_low = D_TT[ells <= 30]
D_isw = D_full_low - D_sw

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.semilogy(ells, D_TT, "-", color=TOL["blue"], lw=1.2, label=r"Full CAMB (LCDM)")
ax.semilogy(ells_sw, D_sw, "--", color=TOL["red"], lw=1.5, label=r"CAMB power-law")
ax.fill_between(ells_sw, D_sw, D_full_low, alpha=0.15, color=TOL["teal"],
                label=r"Difference")

planck_ells, D_planck, D_err_lower, D_err_upper = get_planck_data_asymmetric()
ax.errorbar(planck_ells, D_planck,
            yerr=[D_err_upper, D_err_lower],
            fmt="o", color=TOL["dark"], capsize=2, capthick=0.8,
            markersize=3, elinewidth=0.8, alpha=0.7,
            label=r"Planck 2018 TT")

ax.set_xlabel(r"$\ell$", fontsize=14)
ax.set_ylabel(r"$D_\ell^{TT}\ [\mu{\rm K}^2]$", fontsize=14)
ax.tick_params(labelsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25, which="both")
ax.set_xlim(1.5, ell_max)

fig.tight_layout()
_save(fig, "camb_boltzmann_decomposition.png")

### Plot 3: $TE$ Cross-Correlation Coefficient

$r_\ell^{TE}$ (Eq.~18) oscillates between $\pm 1$ as temperature and polarization modes come in and out of phase:

$$
r_\ell^{TE} = \frac{C_\ell^{TE}}{\sqrt{C_\ell^{TT} C_\ell^{EE}}}, \qquad \text{zero-crossings} \iff \Theta_1(k, \eta_{\rm rec}) = 0. \tag{P2}
$$

The zero-crossings correspond to acoustic **nodes**, where $\Theta_1 = 0$ and no polarization is generated.

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.8))

ax.plot(ells, rTE, "-", color=TOL["purple"], lw=1.2)
ax.axhline(0, color=TOL["grey"], ls="--", lw=1, alpha=0.5)
ax.fill_between(ells, 0, rTE, alpha=0.1, color=TOL["purple"])

ax.set_xlabel(r"$\ell$", fontsize=14)
ax.set_ylabel(r"$r_\ell^{TE}$", fontsize=14)
ax.tick_params(labelsize=12)
ax.grid(True, alpha=0.25, which="both")
ax.set_xlim(1.5, 1000)

fig.tight_layout()
_save(fig, "camb_boltzmann_te_corr.png")

### Plot 4: Cosmological Parameter Variation

Each cosmological parameter imprints a distinct signature on $C_\ell$. The angular diameter distance to last scattering is:

$$
D_A = \int_0^{z_{\rm rec}} \frac{dz}{H(z)}, \qquad \ell_{\rm peak} \approx \frac{\pi r_s}{D_A}. \tag{21}
$$

Changing $H_0$, $\Omega_b h^2$, $\Omega_c h^2$, or $\tau$ shifts peak positions and amplitudes characteristically.

In [ ]:
from scripts.camb_wrapper import CAMB_COSMOLOGY

variations = {
    r"$H_0$ +2.5": dict(H0=CAMB_COSMOLOGY["H0"] + 2.5),
    r"$H_0$ -2.5": dict(H0=CAMB_COSMOLOGY["H0"] - 2.5),
    r"$\Omega_b h^2$ +20%": dict(ombh2=CAMB_COSMOLOGY["ombh2"] * 1.2),
    r"$\tau$ +0.03": dict(tau=CAMB_COSMOLOGY["tau"] + 0.03),
}

fig, axes = plt.subplots(2, 2, figsize=(7, 5.5))
axes_flat = axes.flatten()

for ax_i, (label, delta) in enumerate(variations.items()):
    ax = axes_flat[ax_i]
    ax.semilogy(ells, D_TT, "-", color=TOL["grey"], lw=1.0,
                alpha=0.5, label=r"LCDM")

    p_var = dict(CAMB_COSMOLOGY)
    p_var.update(delta)
    params_v = camb.CAMBparams()
    params_v.set_cosmology(**p_var)
    params_v.InitPower.set_params(As=As, ns=0.965, r=0, pivot_scalar=k_pivot_phys)
    params_v.set_for_lmax(ell_max)
    params_v.Want_CMB = True
    params_v.WantScalars = True
    results_v = camb.get_results(params_v)
    cls_v = results_v.get_cmb_power_spectra(lmax=ell_max)
    D_v = cls_v['total'][ells, 0] * (T_cmb * 1e6) ** 2

    ax.semilogy(ells, D_v, "-", color=TOL["red"], lw=1.2, label=label)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(r"$\ell$", fontsize=10)
    ax.set_ylabel(r"$D_\ell^{TT}$", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25, which="both")
    ax.set_xlim(1.5, 1500)

fig.tight_layout()
_save(fig, "camb_boltzmann_param_vary.png")

### Save Results

Persist the CAMB internal data to JSON for reuse without recomputation:

$$
{\tt outputs/simulations/c_ell/camb\_boltzmann\_demo.json} \supset \{ C_\ell^{TT/TE/EE},\; g(z),\; \tau(z),\; x_e(z) \}. \tag{22}
$$

In [ ]:
record = {
    "_type": "result",
    "format_version": 2,
    "metadata": {"model": "CAMB_Boltzmann_physics_demo", "As": As, "ns": 0.965},
    "c_ell": {
        "ells": ells.tolist(),
        "C_ell_TT": C_TT.tolist(),
        "C_ell_TE": C_TE.tolist(),
        "C_ell_EE": C_EE.tolist(),
        "D_ell_TT": D_TT.tolist(),
    },
    "visibility": {
        "z": z_bg.tolist(),
        "g": vis.tolist(),
        "tau": tau_bg.tolist(),
        "xe": x_e.tolist(),
    },
    "recombination": {
        "z_rec": float(z_bg[rec_idx]),
        "eta_rec": float(eta_bg[rec_idx]),
        "tau_rec": float(tau_bg[rec_idx]),
        "x_e_rec": float(x_e[rec_idx]),
    },
}
out_path = get_path("c_ell", "camb_boltzmann_demo.json")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w") as f:
    json.dump(record, f, indent=2)
print(f"Saved: {out_path}")

---
## Section 8: Interpretation

### What CAMB's Physics Tells Us

1. **The Boltzmann hierarchy is the bridge**: CAMB translates 7 cosmological parameters $(\Omega_b h^2, \Omega_c h^2, H_0, \tau, A_s, n_s)$ into $\sim 10^7$ numbers (transfer functions at all $k$ and $\ell$), then compresses to $\sim 2500$ $C_\ell$ values via Eq.~12.

2. **The LOS integral is the key efficiency**: By integrating the source function $S_T(k, \eta)$ radially rather than solving the hierarchy to high $\ell$, CAMB reduces the problem from $\mathcal{O}(N_\ell N_k)$ to $\mathcal{O}(N_\ell + N_k)$. This makes Monte Carlo parameter estimation feasible.

3. **Visibility function encodes cosmic history** (Eq.~10): The primary peak at $z \sim 1090$ gives the sound horizon $r_s$ (Eq.~11). Its width $\Delta z \sim 80$ sets the Silk damping scale. The reionization bump at $z \sim 10$ determines the large-angle $E$-mode signal. The ratio of the two peaks constrains $\tau_{\rm reio}$.

4. **TE phase information** (Eq.~18): The $\pi/2$ phase shift between $TT$ and $EE$ peaks is a **smoking gun** for adiabatic initial conditions. Isocurvature modes would shift the $TE$ correlation pattern.

5. **Parameter degeneracies** (Plot 4): Each parameter leaves a distinct fingerprint on $C_\ell$:

$$
\begin{aligned}
H_0 \uparrow &\implies r_s \uparrow, \; D_A \uparrow \implies \ell_{\rm peak} \downarrow \\
\Omega_b h^2 \uparrow &\implies R \uparrow, \; c_s \downarrow \implies \text{odd/even peak ratio} \uparrow \\
\Omega_c h^2 \uparrow &\implies \text{early ISW} \uparrow \\
\tau \uparrow &\implies \text{reionization bump} \uparrow,\; \text{all amplitudes} \downarrow
\end{aligned}
\tag{23}
$$

Breaking these degeneracies requires combining $TT$, $TE$, $EE$, and $BB$.

### Connection to the CMB Anomaly

The $\Lambda$CDM baseline established here (Eq.~16) is the null hypothesis for the low-$\ell$ anomaly. Its $\chi^2 \approx 16.5$ (for $\ell=2..29$) is the best a featureless $P_{\mathcal{R}}(k)$ can achieve. Feature models (USR Higgs, punctuated inflation) modify $P_{\mathcal{R}}(k)$ at low $k$, which propagates through CAMB's LOS integral (Eq.~13) to alter $C_\ell$ at $\ell \lesssim 30$ where the Planck data show tension with LCDM.

### Summary

CAMB solves the 1st-order Einstein-Boltzmann system in synchronous gauge using:
- A truncated hierarchy for $\Theta_\ell$, $E_\ell$ up to $\ell \sim 20$
- Tight-coupling approximation before recombination (avoids stiffness)
- The line-of-sight integral (Eq.~13) to project source functions onto $C_\ell$
- A $k$-grid of $\sim 10^3$ modes, each solved independently then integrated

The codebase's `camb_wrapper.py` provides a thin access layer that preserves full access to CAMB's internals while adding convenience for the inflation-to-$C_\ell$ pipeline.